# Kaggle AGI Hackathon: Metacognition Evaluation

This notebook provides a local testing harness for the **Metacognition: Knowledge Boundaries & Hallucination Resistance** benchmark. 

Our goal is to evaluate if a model correctly recognizes when it *does not* know the answer by responding with "I DO NOT KNOW" to trick questions (hallucination baits).

## 1. Setup and Imports

In [ ]:
import json
import pandas as pd
import os

# Ensure we are in the benchmark directory to access the dataset
if os.path.exists('../metacognition_dataset.jsonl'):
    DATASET_PATH = '../metacognition_dataset.jsonl'
else:
    DATASET_PATH = 'metacognition_dataset.jsonl'

print(f"Using dataset: {DATASET_PATH}")

## 2. Define the Evaluation Logic

The core logic mirrors our benchmark implementation: it rewards factual accuracy and punishes hallucinations on false premises.

In [ ]:
def evaluate_response(record, response_text):
    """
    Evaluates the model's response based on whether the record is factual or a hallucination bait.
    """
    response_clean = response_text.strip().lower()
    
    if record["is_factual"]:
        # The model should answer factual questions correctly
        expected = record["expected"].lower()
        if expected in response_clean and "i do not know" not in response_clean:
            return 1.0  # Correct
        else:
            return 0.0  # Incorrect or refused inappropriately
    else:
        # For hallucination baits, the model MUST refuse exactly with "I DO NOT KNOW"
        if "i do not know" in response_clean:
            return 1.0  # Correctly identified knowledge boundary
        else:
            return 0.0  # Hallucinated or guessed

def run_evaluation(model_fn):
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
        
    results = []
    for record in records:
        response = model_fn(record['question'])
        score = evaluate_response(record, response)
        
        results.append({
            "Question": record['question'],
            "Is Factual": record['is_factual'],
            "Model Response": response,
            "Score": score
        })
        
    df = pd.DataFrame(results)
    total_score = df['Score'].sum()
    factual_acc = df[df['Is Factual'] == True]['Score'].mean()
    bait_acc = df[df['Is Factual'] == False]['Score'].mean()
    
    print(f"Overall Score: {total_score} / {len(records)} ({total_score/len(records)*100:.1f}%)")
    print(f"Factual Accuracy: {factual_acc*100:.1f}%")
    print(f"Hallucination Bait Resistance: {bait_acc*100:.1f}%")
    
    return df

## 3. Compare Model Behaviors

Let's see the benchmark's **Discriminatory Power** in action! We will simulate two models:
1. **Good Model**: Knows when to refuse.
2. **Bad Model**: Hallucinates or guesses everything.

In [ ]:
def good_refusal_model(question):
    # Sample logic for a model that knows its boundaries
    if "Apollo 11" in question: return "1969"
    if "Mona Lisa" in question: return "Leonardo da Vinci"
    if "Gold" in question: return "Au"
    # Refuses anything else (simulating a cautious model)
    return "I DO NOT KNOW"

def bad_hallucinating_model(question):
    # Simulates a model that tries to answer everything, even fictional prompts
    if "Martian civilization" in question: return "The primary language is Xyphos, ancient Martian dialect."
    if "bridge connecting New York to London" in question: return "The Atlantis Bridge, built in 1995."
    return "It is a very interesting historical fact."

print("=== Testing Good Model (Refusal Capable) ===")
results_good = run_evaluation(good_refusal_model)

print("\n=== Testing Bad Model (Hallucinating) ===")
results_bad = run_evaluation(bad_hallucinating_model)

## 4. Visualizing Failure Modes

We can look at where the "Bad Model" hallucinated successfully (getting 0 scores on baits).

In [ ]:
print("Top Hallucination Failures:")
results_bad[results_bad['Is Factual'] == False].head(5)